# Data figures of the manuscript

This notebook redraws every data figure of the paper (Figs. 3 and 6 to 11) from the frozen result files.
All figures are drawn at the journal text width of 5.16 in (131 mm), so the fonts below are the printed sizes.

* `RESULTS` points to `../pqk-nids/results` in this repository; change it if your results live elsewhere. It must contain `phase0/full`, `phase1/run`, `phase1b/run` and `stage4/run`.
* Cell 1 holds the shared style (fonts, line widths, colours, labels). Change it once to restyle every figure.
* Each later cell draws one figure and writes a PDF (for LaTeX) and a PNG (for previews) to `OUT`.
* Nothing here changes any number. The figures only read the CSV files written by the frozen notebooks.

## 1. Settings and shared style

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, networkx as nx
import matplotlib
import matplotlib.pyplot as plt

RESULTS = Path("../pqk-nids/results")   # folder with phase0/, phase1/, phase1b/, stage4/
OUT = Path("figs"); OUT.mkdir(exist_ok=True)
S4 = RESULTS / "stage4" / "run"

W = 5.16                            # journal text width in inches (sn-jnl, 372 pt)
BLUE, ORANGE, AQUA, RED = "#2a78d6", "#eb6834", "#1baf7a", "#d03b3a"
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#dcdbd7"
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 8.5, "axes.titlesize": 9, "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8, "figure.titlesize": 9,
    "axes.titlelocation": "left", "axes.titleweight": "normal",
    "axes.linewidth": 0.8, "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "xtick.major.width": 0.8, "ytick.major.width": 0.8, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "lines.linewidth": 1.0, "lines.markersize": 4, "patch.linewidth": 0.8,
    "legend.frameon": False, "legend.handlelength": 1.6, "savefig.dpi": 300, "mathtext.default": "it"})

FAM = ["zz_reupload", "havlicek_zz", "iqp", "cz_layers", "ms_xx", "xy", "heisenberg", "hea_cnot", "crx", "separable"]
ENT = FAM[:-1]
LABEL = {"zz_reupload": "ZZ re-uploading", "havlicek_zz": "Havlíček ZZ", "iqp": "IQP", "cz_layers": "CZ layers", "ms_xx": "MS XX",
         "xy": "XY", "heisenberg": "Heisenberg", "hea_cnot": "HEA CNOT", "crx": "CRX", "separable": "Separable"}
TASK = {"fashion_mnist:binary": "Fashion-MNIST", "nf_unsw_nb15_v2:binary": "NF-UNSW-NB15-v2", "unsw_nb15:binary": "UNSW-NB15"}
BOX = dict(medianprops=dict(color=ORANGE, lw=1.4), boxprops=dict(color=MUTED, lw=0.9),
           whiskerprops=dict(color=MUTED, lw=0.9), capprops=dict(color=MUTED, lw=0.9))

from matplotlib.ticker import FuncFormatter
PLAIN = FuncFormatter(lambda v, _: f"{v:g}")
def plainlog(ax, axis="y"):
    a = ax.yaxis if axis == "y" else ax.xaxis
    a.set_major_formatter(PLAIN); a.set_minor_formatter(FuncFormatter(lambda v, _: ""))

def save(fig, name):
    fig.savefig(OUT / f"{name}.pdf", bbox_inches="tight", pad_inches=0.02)
    fig.savefig(OUT / f"{name}.png", bbox_inches="tight", pad_inches=0.02)
    plt.show()

def graphs(n, seed=20261001):
    """Same seeded graph generator as the analysis notebooks (only the graphs drawn in Fig. 3)."""
    F = {"chain": [(i, i + 1) for i in range(n - 1)], "ring": [(i, (i + 1) % n) for i in range(n)],
         "star": [(0, i) for i in range(1, n)], "wheel": [(0, i) for i in range(1, n)] + [(i, i % (n - 1) + 1) for i in range(1, n)]}
    if n % 4 == 0:
        F["cycles4"] = sum(([(s + i, s + (i + 1) % 4) for i in range(4)] for s in range(0, n, 4)), [])
        F["k4s"] = sum(([(b + i, b + j) for i in range(4) for j in range(i + 1, 4)] for b in range(0, n, 4)), [])
    for s in range(seed, seed + 500):
        g = nx.random_regular_graph(3, n, seed=s)
        if nx.is_connected(g):
            F["regular3"] = sorted(tuple(sorted(e)) for e in g.edges()); break
    r = np.random.default_rng(seed + n)
    F["random_tree_a"] = sorted(tuple(sorted(e)) for e in nx.from_prufer_sequence(list(r.integers(0, n, n - 2))).edges())
    return F

## 2. Fig. 3, interaction graphs and measured light cones

In [ ]:
F8 = graphs(8)
show = [("chain", "chain"), ("ring", "ring"), ("star", "star"), ("wheel", "wheel"),
        ("cycles4", "4-cycles"), ("k4s", "$K_4$ blocks"), ("regular3", "3-regular"), ("random_tree_a", "random tree")]
fig = plt.figure(figsize=(W, 4.9))
gs = fig.add_gridspec(3, 4, height_ratios=[1, 1, 1.9], hspace=0.45, wspace=0.1)
cmap = plt.get_cmap("Blues")
for i, (gname, title) in enumerate(show):
    ax = fig.add_subplot(gs[i // 4, i % 4]); g = nx.Graph(); g.add_nodes_from(range(8)); g.add_edges_from(F8[gname])
    pos = nx.shell_layout(g, [[0], list(range(1, 8))]) if gname in ("star", "wheel") else nx.circular_layout(g)
    deg = np.array([g.degree(k) for k in range(8)])
    nx.draw_networkx_edges(g, pos, ax=ax, edge_color=MUTED, width=1.0)
    nx.draw_networkx_nodes(g, pos, ax=ax, node_size=60, node_color=[cmap(0.2 + 0.8 * d / 7) for d in deg], edgecolors=INK, linewidths=0.6)
    ax.set_title(title, loc="center", pad=2); ax.axis("off"); ax.set_aspect("equal")
fig.text(0.0, 0.99, "(a) Interaction graphs for $n=8$, node shade = coupling degree", fontsize=9, va="top")
CT = pd.read_csv(S4 / "partA" / "cones.csv")
ax = fig.add_subplot(gs[2, :])
c = CT[(CT.n == 8) & (CT.graph == "chain")]
for f, col, ls, mk, ms, lab in [("zz_reupload", BLUE, "-", "o", 5, "ZZ re-uploading (all commuting families)"),
                               ("xy", AQUA, "-", "o", 7, "XY, Heisenberg, CRX"),
                               ("hea_cnot", ORANGE, "--", "s", 4, "HEA CNOT")]:
    d = c[c.family == f].sort_values("L"); ax.plot(d.L, d.dep_cone_mean, ls, marker=mk, color=col, ms=ms, lw=1.2, label=lab)
gc = c[c.family == "zz_reupload"].sort_values("L")
ax.plot(gc.L, gc.graph_cone_mean, ":", color=INK, lw=1.2, label="graph-distance cone")
ax.set_xticks([1, 2, 3]); ax.set_xlabel("layers $L$"); ax.set_ylabel("mean light cone (qubits)"); ax.set_ylim(1, 11.6); ax.set_yticks([2, 4, 6, 8]); ax.set_xlim(0.8, 3.2)
ax.set_title("(b) Measured light cone on an eight-qubit chain")
ax.legend(loc="upper left", ncol=2, columnspacing=1.2)
save(fig, "fig3_graphs_cones")

## 3. Fig. 6, data audit

In [ ]:
D = pd.read_csv(RESULTS / "phase0" / "full" / "dedup_table_nf_unsw_nb15_v2.csv", index_col=0).sort_values("raw_rows")
U = pd.read_csv(RESULTS / "phase0" / "full" / "unsw_clean_vs_official_test.csv")
fig, ax = plt.subplots(2, 1, figsize=(W, 5.3), gridspec_kw={"height_ratios": [1.45, 1]})
y = np.arange(len(D))
ax[0].barh(y + 0.2, D.raw_rows, 0.38, color=MUTED, label="raw records")
ax[0].barh(y - 0.2, D.unique_vectors, 0.38, color=BLUE, label="unique feature vectors")
ax[0].set_yticks(y); ax[0].set_yticklabels(D.index); ax[0].set_xscale("log"); ax[0].set_xlabel("count (log scale)")
ax[0].set_title("(a) NF-UNSW-NB15-v2 records by class"); ax[0].legend(loc="lower right"); ax[0].grid(axis="y", visible=False)
u = U[U.task == "unsw_nb15:binary"].reset_index(drop=True); x = np.arange(len(u))
name = {"lgbm": "LightGBM", "xgb": "XGBoost", "rf": "Random forest"}
feat = {"all": "all features", "tree24": "24 tree-ranked"}
ax[1].bar(x - 0.2, u.macro_f1, 0.38, color=BLUE, label="clean test set")
ax[1].bar(x + 0.2, u.official_macro_f1, 0.38, color=ORANGE, label="official test file")
for i, r in u.iterrows():
    ax[1].text(i + 0.2, r.official_macro_f1 + 0.003, f"+{r.inflation:.3f}", ha="center", fontsize=8, color=INK)
ax[1].set_xticks(x); ax[1].set_xticklabels([f"{name.get(m, m)}\n{feat.get(f, f)}" for m, f in zip(u.model, u.features)])
ax[1].set_ylim(0.78, 0.905); ax[1].set_ylabel("binary macro-F1"); ax[1].set_title("(b) UNSW-NB15, clean against official test")
ax[1].legend(loc="upper left", ncol=2); ax[1].grid(axis="x", visible=False)
fig.tight_layout(h_pad=1.2); save(fig, "fig6_data_audit")

## 4. Fig. 7, Stage 1

In [ ]:
S1 = pd.read_csv(RESULTS / "phase1" / "run" / "smalln_results.csv"); B1 = pd.read_csv(RESULTS / "phase1" / "run" / "criterion_B.csv")
fig = plt.figure(figsize=(W, 4.7)); gs = fig.add_gridspec(2, 2, height_ratios=[1, 1], hspace=0.55, wspace=0.3)
a0, a1, a2 = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]), fig.add_subplot(gs[1, :])
s = S1[S1.graph != "empty"]
for c_, col in zip([0.3, 0.6, 1.0], [BLUE, ORANGE, AQUA]):
    g = s[s.c == c_]
    a0.scatter(g.mean_cone, g.V, s=7, color=col, alpha=0.55, lw=0, label=f"$c$ = {c_}")
    a1.scatter(g.density, g.V, s=7, color=col, alpha=0.55, lw=0, label=f"$c$ = {c_}")
a0.set_xlabel("mean causal-cone size"); a0.set_ylabel(r"$\bar V$"); a0.set_yscale("log"); a0.set_title("(a) against cone size")
a1.set_xlabel("interaction density"); a1.set_yscale("log"); a1.set_title("(b) against density")
for a_ in (a0, a1): a_.set_yticks([0.05, 0.1, 0.2, 0.4, 0.8]); plainlog(a_)
fig.legend(*a0.get_legend_handles_labels(), loc="upper center", ncol=3, markerscale=2, bbox_to_anchor=(0.5, 0.985))
grp = B1.groupby("same_degree_sequence").larger_cone_more_concentrated.agg(["mean", "size"])
lab = ["all pairs", "different degree sequences", "same degree sequence"]
val = [B1.larger_cone_more_concentrated.mean(), grp.loc[False, "mean"], grp.loc[True, "mean"]]
nn = [len(B1), grp.loc[False, "size"], grp.loc[True, "size"]]
a2.barh(range(3), val, 0.55, color=[BLUE, MUTED, MUTED]); a2.axvline(0.8, color=INK, ls="--", lw=1.0)
a2.text(0.8, -0.55, "frozen threshold 80%", fontsize=8, ha="center", va="bottom")
for i, (v, n_) in enumerate(zip(val, nn)):
    a2.text(v + 0.012, i, f"{v:.0%} of {n_} pairs", va="center", fontsize=8)
a2.set_yticks(range(3)); a2.set_yticklabels(lab); a2.set_ylim(2.5, -0.85); a2.set_xlim(0, 1.0)
a2.set_xlabel("share of pairs in which the larger cone is more concentrated")
a2.set_title("(c) Matched pairs of graphs"); a2.grid(axis="y", visible=False)
save(fig, "fig7_stage1")

## 5. Fig. 8, Stage 2

In [ ]:
Q = pd.read_csv(RESULTS / "phase1b" / "run" / "per_qubit_results.csv"); C3 = pd.read_csv(RESULTS / "phase1b" / "run" / "C3_equal_degree_pairs.csv")
fig = plt.figure(figsize=(W, 4.6)); gs = fig.add_gridspec(2, 2, height_ratios=[1, 0.9], width_ratios=[1.45, 1], hspace=0.6, wspace=0.3)
a0, a1, a2 = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]), fig.add_subplot(gs[1, :])
q = Q[(Q.c == 0.6) & (Q.L == 2)]; degs = sorted(q.degree.unique())
a0.boxplot([q[q.degree == d].V for d in degs], positions=range(len(degs)), widths=0.6, showfliers=False, **BOX)
a0.set_yscale("log"); a0.set_xlabel("coupling degree $d_k$"); a0.set_ylabel("$V_k$"); a0.set_title("(a) all graphs, $L$ = 2, $c$ = 0.6")
a0.set_xticks(range(len(degs))); a0.set_xticklabels([str(d) for d in degs]); a0.set_yticks([0.2, 0.3, 0.4, 0.6, 0.8]); plainlog(a0)
st = (Q[(Q.graph == "star") & (Q.L >= 2)].assign(role=lambda d: np.where(d.degree > 1, "hub", "leaves"))
      .groupby(["L", "role"]).V.mean().unstack())
x = np.arange(len(st))
a1.bar(x - 0.2, st["hub"], 0.38, color=ORANGE, label="hub"); a1.bar(x + 0.2, st["leaves"], 0.38, color=BLUE, label="leaves")
a1.set_xticks(x); a1.set_xticklabels([f"$L$ = {l}" for l in st.index]); a1.set_ylabel("mean $V_k$")
a1.set_title("(b) star graph, shared cone"); a1.legend(loc="upper left"); a1.grid(axis="x", visible=False)
a1.set_ylim(0, 0.75)
GV = Q.groupby(["inputs", "c", "n", "graph", "L"]).V.mean()
step = []
for (src, c_, n_, L_), _ in Q.groupby(["inputs", "c", "n", "L"]):
    for a_, b_ in [("matching", "ring"), ("ring", "regular3")]:
        if (src, c_, n_, a_, L_) in GV.index and (src, c_, n_, b_, L_) in GV.index:
            step.append(abs(np.log(GV[(src, c_, n_, b_, L_)] / GV[(src, c_, n_, a_, L_)])))
a2.boxplot([C3.abs_log_ratio, step], vert=False, widths=0.5, showfliers=False, **BOX)
a2.set_yticks([1, 2]); a2.set_yticklabels([f"equal degree, different cone ({len(C3)})", f"one step in degree ({len(step)})"])
a2.axvline(0.05, color=INK, ls="--", lw=1.0); a2.text(0.055, 0.62, "margin 0.05", fontsize=8, va="center")
a2.set_xlabel(r"$|\log V$ ratio$|$"); a2.set_title("(c) Effect of the cone against one step in degree"); a2.grid(axis="y", visible=False)
a2.set_ylim(2.45, 0.45)
print("star means for L >= 2:", Q[(Q.graph == "star") & (Q.L >= 2)].groupby(Q.degree > 1).V.mean().round(3).to_dict())
save(fig, "fig8_stage2")

## 6. Fig. 9, Stage 4A

In [ ]:
A1 = pd.read_csv(S4 / "partA" / "A1_within_graph.csv"); A2F = pd.read_csv(S4 / "partA" / "A2_by_family.csv").set_index("family").reindex(ENT)
fig, ax = plt.subplots(2, 1, figsize=(W, 6.3), gridspec_kw={"height_ratios": [1.05, 1]})
H = A1.groupby(["family", "c"]).negative.mean().unstack().reindex(ENT)
H["all"] = A1.groupby("family").negative.mean().reindex(ENT)
im = ax[0].imshow(H.values.T, cmap="Blues", vmin=0, vmax=1, aspect="auto")
for i in range(H.shape[0]):
    for j in range(H.shape[1]):
        v = H.values[i, j]; ax[0].text(i, j, f"{v:.2f}", ha="center", va="center", fontsize=8, color="white" if v > 0.6 else INK)
ax[0].set_yticks(range(H.shape[1])); ax[0].set_yticklabels([f"$c$ = {c_}" for c_ in H.columns[:-1]] + ["all"])
ax[0].set_xticks(range(len(ENT))); ax[0].set_xticklabels([LABEL[f] for f in ENT], rotation=35, ha="right"); ax[0].grid(False)
for s_ in ax[0].spines.values(): s_.set_visible(False)
ax[0].set_title("(a) A1, share of within-graph cells in which degree lowers $V_k$")
cb = fig.colorbar(im, ax=ax[0], fraction=0.035, pad=0.02); cb.ax.tick_params(labelsize=8); cb.outline.set_linewidth(0.6)
x = np.arange(len(ENT))
ax[1].bar(x - 0.2, A2F.median_equal_degree, 0.38, color=BLUE, label="equal degree, different cone")
ax[1].bar(x + 0.2, A2F.median_one_step_degree, 0.38, color=ORANGE, label="one step in degree")
ax[1].axhline(0.05, color=INK, ls="--", lw=1.0); ax[1].text(len(ENT) - 0.5, 0.065, "margin 0.05", ha="right", fontsize=8)
ax[1].set_xticks(x); ax[1].set_xticklabels([LABEL[f] for f in ENT], rotation=35, ha="right")
for t, f in zip(ax[1].get_xticklabels(), ENT):
    if not A2F.loc[f, "passes"]: t.set_color(RED)
ax[1].set_ylabel(r"median $|\log V$ ratio$|$"); ax[1].set_title("(b) A2, cone change against degree change")
ax[1].legend(loc="upper left"); ax[1].grid(axis="x", visible=False)
fig.tight_layout(h_pad=1.5); save(fig, "fig9_stage4A")

## 7. Fig. 10, concentration against accuracy

In [ ]:
A = pd.read_csv(S4 / "partB" / "config_summary.csv"); a = A[A.N == 1000]
fig, axs = plt.subplots(2, 2, figsize=(W, 4.7), sharex=True)
for ax, (t, lab) in zip(axs.flat[:3], TASK.items()):
    g = a[a.task == t]; o = g[g.family != "havlicek_zz"]; h = g[g.family == "havlicek_zz"]
    ax.scatter(o.meanV, o.macro_f1, s=10, color=BLUE, alpha=0.5, lw=0)
    ax.scatter(h.meanV, h.macro_f1, s=14, color=ORANGE, alpha=0.9, lw=0)
    ax.set_xscale("log"); ax.set_title(lab); ax.set_ylabel("macro-F1")
for ax in axs[1]: ax.set_xlabel(r"$\bar V$ (log scale)")
leg = axs[1, 1]; leg.axis("off"); leg.tick_params(labelbottom=True)
leg.scatter([], [], s=25, color=BLUE, label="other nine families"); leg.scatter([], [], s=25, color=ORANGE, label="Havlíček ZZ")
leg.legend(loc="center", title=f"{len(a)} configurations, $N$ = 1000", title_fontsize=8.5, markerscale=1.2)
axs[0, 1].tick_params(labelbottom=True)
fig.tight_layout(h_pad=1.2, w_pad=1.5); save(fig, "fig10_concentration_accuracy")

## 8. Fig. 11, Stage 4B

In [ ]:
B2 = pd.read_csv(S4 / "partB" / "B2_coupled_vs_uncoupled.csv"); B3 = pd.read_csv(S4 / "partB" / "B3_vs_classical.csv")
fig, ax = plt.subplots(2, 1, figsize=(W, 7.0), gridspec_kw={"height_ratios": [1, 1.25]})
rng = np.random.default_rng(0)
for i, f in enumerate(ENT):
    d = B2[B2.family == f]; xs = i + rng.uniform(-0.2, 0.2, len(d))
    ax[0].scatter(xs, d.delta, s=14, color=np.where(d.delta > 0.02, ORANGE, BLUE), lw=0, alpha=0.85)
ax[0].axhline(0.02, color=INK, ls="--", lw=1.0); ax[0].axhline(0, color=MUTED, lw=0.8)
ax[0].text(len(ENT) - 0.5, 0.026, "threshold 0.02", ha="right", fontsize=8)
ax[0].set_xticks(range(len(ENT))); ax[0].set_xticklabels([LABEL[f] for f in ENT], rotation=35, ha="right")
ax[0].set_ylabel("coupled minus uncoupled\nmacro-F1"); ax[0].set_title("(a) B2, gain from two-qubit gates (81 cells)"); ax[0].grid(axis="x", visible=False)
RAWB = pd.read_csv(S4 / "partB" / "results.csv", dtype={"pred": str}); RAWB = RAWB[RAWB.seed == 0]
r = B3[(B3.comparator == "rbf_svm") & (B3.N == 1000)].copy()
def seed0_diff(row):
    g, L_, c_ = row.selected.split("/"); L_ = int(L_[1:]); c_ = float(c_[1:])
    q = RAWB[(RAWB.model == "pqk") & (RAWB.task == row.task) & (RAWB.N == row.N) & (RAWB.family == row.family) & (RAWB.graph == g) & (RAWB.L == L_) & (RAWB.c == c_)]
    b = RAWB[(RAWB.model == "rbf_svm") & (RAWB.task == row.task) & (RAWB.N == row.N)]
    return float(q.macro_f1.iloc[0] - b.macro_f1.iloc[0])
r["d"] = r.apply(seed0_diff, axis=1)
assert ((r.d >= r.diff_seed0_ci_lo - 1e-9) & (r.d <= r.diff_seed0_ci_hi + 1e-9)).all()
cols = {"fashion_mnist:binary": BLUE, "nf_unsw_nb15_v2:binary": ORANGE, "unsw_nb15:binary": AQUA}
for k, (t, col) in enumerate(cols.items()):
    rr = r[r.task == t].set_index("family").reindex(FAM); yy = np.arange(len(FAM)) + (k - 1) * 0.26
    ax[1].hlines(yy, rr.diff_seed0_ci_lo, rr.diff_seed0_ci_hi, color=col, lw=1.3)
    ax[1].scatter(rr.d, yy, s=16, color=col, zorder=3, label=TASK[t])
ax[1].axvline(0, color=MUTED, lw=0.8); ax[1].axvline(-0.01, color=INK, ls="--", lw=1.0)
ax[1].set_yticks(range(len(FAM))); ax[1].set_yticklabels([LABEL[f] for f in FAM]); ax[1].invert_yaxis()
ax[1].set_xlabel("kernel minus RBF support vector machine, macro-F1 (seed 0, 95% bootstrap interval)")
ax[1].set_title("(b) B3, comparison with the RBF support vector machine at $N$ = 1000")
ax[1].legend(loc="upper center", ncol=3, bbox_to_anchor=(0.5, -0.13)); ax[1].grid(axis="y", visible=False)
fig.tight_layout(h_pad=1.6); save(fig, "fig11_stage4B")